In [0]:
spark.sql("""create database if not exists freshcart_team1""")


In [0]:
from pyspark.sql import functions as f
import uuid 
    

In [0]:
CATALOG='de_workspace26'
SCHEMA = "freshcart_team1"
files = [
    "customers",
    "deliveries",
    "order_items",
    "orders",
    "products"
]
# BASE_PATH = "s3://s3-de-q1-26/DE-Training/freshcart-datalake-team1-db/raw"  
BASE_PATH="s3://s3-de-q1-26/DE-Training/freshcart-datalake-team1/streaming"
CHECKPOINT_BASE = "s3://s3-de-q1-26/DE-Training/freshcart-datalake-team1-db/checkpoints/bronze"


run_id = str(uuid.uuid4())

# df_dict={}

# first we read all the raw daya from volume and add some MetaData Coloumns on it 
queries = [] 

for file in files:
    file_path = f"{BASE_PATH}/{file}/"
    checkpoint_path = f"{CHECKPOINT_BASE}/{file}/checkpoints"
    schema_path = f"{CHECKPOINT_BASE}/{file}/schema"

    table_name = f"{CATALOG}.{SCHEMA}.bronze_data_{file}"
    output_path = f"s3://s3-de-q1-26/DE-Training/freshcart-datalake-team1-db/bronze/{file}/"

    print(f"Processing: {file}")

    
    # spark.sql(f"""
    #     CREATE TABLE IF NOT EXISTS {table_name}
    #     USING DELTA
    #     LOCATION '{output_path}'
    # """)

   
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)  
        .option("header", "true")
        .option("inferSchema", "true")
        .load(file_path)
    )

    df = (
        df.withColumn("_source", f.lit("s3"))
          .withColumn("_ingest_ts", f.current_timestamp())
          .withColumn("_file_name", f.lit(file))
          .withColumn("_run_id", f.lit(run_id))
          .withColumn("ingest_date", f.to_date(f.current_timestamp()))
    )


    query=(
        df.writeStream
          .format("delta")
          .option("checkpointLocation", checkpoint_path)
          .option("mergeSchema", "true")
          .option('path', output_path)
          .partitionBy("ingest_date")
          .outputMode("append")
          .trigger(availableNow=True)
          .toTable(table_name)
    )

    queries.append(query)

    print(f"Writing to table: {table_name} is done")



for query in queries:
    query.awaitTermination()

print("Bronze ingestion complete for all tables!")   

    

   

In [0]:
%sql
DELETE FROM de_workspace26.freshcart_team1.bronze_data_orders
WHERE order_id IS NULL 
   OR TRY_CAST(total_amount AS DOUBLE) < 0;

In [0]:
# %sql


# alter table de_workspace26.freshcart_team1.bronze_data_orders
# add constraint validate_order_id check(order_id IS NOT NULL);

# --  this will throw error because total_amount is string type will fix these things in silver so wait till silver
# alter table de_workspace26.freshcart_team1.bronze_orders
# add constraint valid_payments check(total_amount>=0)


In [0]:
%sql

show tables in freshcart_team1;

In [0]:
%sql
select count(*) as orders_count from de_workspace26.freshcart_team1.bronze_data_orders;










In [0]:
%sql
select count(*) as deliveries_count from de_workspace26.freshcart_team1.bronze_data_deliveries;

In [0]:
%sql
select count(*) as order_items_count from  de_workspace26.freshcart_team1.bronze_data_order_items;

In [0]:
%sql

select count(*) as customer_counts from  de_workspace26.freshcart_team1.bronze_data_customers;

In [0]:
%sql
select count(*) as products_count from de_workspace26.freshcart_team1.bronze_data_products;

In [0]:
df = spark.read.table("de_workspace26.freshcart_team1.bronze_data_customers")

df.show()




In [0]:
df_bronze_order=spark.read.format('delta').table('freshcart_team1.bronze_data_orders')

df_bronze_order.show(5)

In [0]:
df_bronze_order.filter(f.col('status')=='DELIVERED').select('order_id','customer_id','total_amount').explain(True)

In [0]:
# %sql


# ALTER TABLE de_workspace26.freshcart_team1.bronze_orders
# DROP CONSTRAINT validate_order_id;

# ALTER TABLE de_workspace26.freshcart_team1.bronze_orders
# DROP CONSTRAINT valid_payments;

In [0]:
%sql
-- select count(*) from freshcart_team1.bronze_orders;


select count(*) from freshcart_team1.bronze_data_orders;

In [0]:
%sql
describe detail freshcart_team1.bronze_data_orders

In [0]:


# FILES = [
#     "customers",
# #     "deliveries", 
# #     "order_items",
# #     "orders",
# #     "products"
# ]

# CHECKPOINT_BASE = "s3://s3-de-q1-26/DE-Training/freshcart-datalake-team1-db/checkpoints/silver"

# for file in FILES:
#     checkpoint_path = f"{CHECKPOINT_BASE}/{file}/checkpoints"
#     # schema_path     = f"{CHECKPOINT_BASE}/{file}/schema"
    
#     # clear checkpoints
#     dbutils.fs.rm(checkpoint_path, recurse=True)
#     # dbutils.fs.rm(schema_path, recurse=True)
    
#     print(f"✅ Cleared checkpoints for: {file}")

# print("All checkpoints cleared! Now run your ingestion code.")

In [0]:
# FILES = ["customers", "deliveries", "order_items", "orders", "products"]

# OUTPUT_BASE = "s3://s3-de-q1-26/DE-Training/freshcart-datalake-team1-db/bronze"

# for file in FILES:
#     output_path = f"{OUTPUT_BASE}/{file}/"
    
#     dbutils.fs.rm(output_path, recurse=True)
    
#     print(f"✅ Cleared delta logs for: {file}")

# print("🎉 Done! Now run your ingestion code.")

In [0]:
# %sql
# DROP TABLE IF EXISTS de_workspace26.freshcart_team1.bronze_data_customers;
# DROP TABLE IF EXISTS de_workspace26.freshcart_team1.bronze_data_deliveries;
# DROP TABLE IF EXISTS de_workspace26.freshcart_team1.bronze_data_order_items;
# DROP TABLE IF EXISTS de_workspace26.freshcart_team1.bronze_data_orders;
# DROP TABLE IF EXISTS de_workspace26.freshcart_team1.bronze_data_products;